## Exercise 7 - Testing Tableau

In [48]:
import geopandas as gpd
import intake
import pandas as pd


import calitp
from calitp.tables import tbl
from calitp import query_sql
from siuba import *

import shapely
from shapely.geometry import Polygon, LineString, Point
from shapely import wkt
from shapely.geometry import Point, Polygon
from shapely.wkt import loads
import shared_utils 
import os
from calitp.tables import tbl

os.environ["CALITP_BQ_MAX_BYTES"] = str(50_000_000_000)

WGS84 = "EPSG:4326"
CA_StatePlane = "EPSG:2229"  # units are in feet
CA_NAD83Albers = "EPSG:3310"  # units are in meters

SQ_MI_PER_SQ_M = 3.86 * 10 ** -7
FEET_PER_MI = 5_280
SQ_FT_PER_SQ_MI = 2.788 * 10 ** 7


## A few Bay Area agencies
[Agencies.yml](https://github.com/cal-itp/data-infra/blob/main/airflow/data/agencies.yml)
* AC Transit: 4
* BART: 279
* SF Muni: 282 
* Samtrans: 290
*  Santa Clara Valley Transportation Authority: 294

In [49]:
#choosing a different set of ITP IDS
ITP_ID = [4,279,282,290,294]

In [50]:
bay_area = (
    tbl.views.gtfs_schedule_dim_stops()
    >> select(_.itp_id == _.calitp_itp_id, _.stop_id, 
             _.stop_lat, _.stop_lon, _.stop_name)
    >> arrange(_.itp_id, _.stop_id, 
               _.stop_lat, _.stop_lon)
    >> collect()
    >> filter(_.itp_id.isin(ITP_ID))
    >> distinct()
)

In [51]:
bay_area.head(2)

,itp_id,stop_id,stop_lat,stop_lon,stop_name
0,4,10,37.742485,-122.250108,Aughinbaugh Way & Robert Davey Jr Dr
1,4,100,37.759356,-122.251237,2217 Otis Dr


In [52]:
bay_area.columns

Index(['itp_id', 'stop_id', 'stop_lat', 'stop_lon', 'stop_name'], dtype='object')

In [53]:
f'A total of {len(bay_area)} rows'

'A total of 27951 rows'

In [54]:
bay_area2 = bay_area.drop_duplicates(subset=['stop_id','itp_id'])

In [55]:
f'only {len(bay_area2)} rows after dropping duplicates'

'only 25010 rows after dropping duplicates'

In [56]:
#creating point geometry from longtitude &  latitude 
bay_area2 = shared_utils.geography_utils.create_point_geometry(bay_area2, 
                                                               'stop_lon', 'stop_lat')

In [57]:
bay_area2.head(2)

,itp_id,stop_id,stop_lat,stop_lon,stop_name,geometry
0,4,10,37.742485,-122.250108,Aughinbaugh Way & Robert Davey Jr Dr,POINT (-122.25011 37.74249)
1,4,100,37.759356,-122.251237,2217 Otis Dr,POINT (-122.25124 37.75936)


In [58]:
def make_routes_shapefile(ITP_ID_LIST=[], CRS="EPSG:4326", alternate_df=None):
    """
    Parameters:
    ITP_ID_LIST: list. List of ITP IDs found in agencies.yml
    CRS: str. Default is WGS84, but able to re-project to another CRS.
    Returns a geopandas.GeoDataFrame, where each line is the operator-route-line geometry.
    """

    all_routes = gpd.GeoDataFrame()

    for itp_id in ITP_ID_LIST:
        if alternate_df is None:
            shapes = (
                tbl.gtfs_schedule.shapes()
                >> filter(_.calitp_itp_id == int(itp_id))
                >> collect()
            )

        elif alternate_df is not None:
            shapes = alternate_df.copy()

            # With historical gtfs_schedule_type2.shapes table, there is a valid shape_id
            if shapes.shape_id.isnull().sum() == 0:
                pass
            else:
                # shape_id is None, which will throw up an error later on when there's groupby
                # So, create shape_id and fill it in with route_id info, but only if it's missing
                # To flag these, shape_id as a column wouldn't even exist.
                if "shape_id" not in shapes.columns:
                    shapes = shapes.assign(
                        shape_id=shapes.route_id,
                    )
                else:
                    # This handles operators where most of their shape_ids are valid
                    # Let's just drop missing ones.
                    shapes = shapes[shapes.shape_id.notna()].reset_index(drop=True)

        # Make a gdf
        shapes = gpd.GeoDataFrame(
            shapes,
            geometry=gpd.points_from_xy(shapes.shape_pt_lon, shapes.shape_pt_lat),
            crs=WGS84,
        )

        # Count the number of stops for a given shape_id
        # Must be > 1 (need at least 2 points to make a line)
        shapes = shapes.assign(
            num_stops=(
                shapes.groupby("shape_id")["shape_pt_sequence"].transform("count")
            )
        )

        # Drop the shape_ids that can't make a line
        shapes = shapes[shapes.num_stops > 1].reset_index(drop=True)

        # Now, combine all the stops by stop sequence, and create linestring
        for route in shapes.shape_id.unique():
            single_shape = (
                shapes
                >> filter(_.shape_id == route)
                >> mutate(shape_pt_sequence=_.shape_pt_sequence.astype(int))
                # arrange in the order of stop sequence
                >> arrange(_.shape_pt_sequence)
            )

            # Convert from a bunch of points to a line (for a route, there are multiple points)
            route_line = shapely.geometry.LineString(list(single_shape["geometry"]))
            single_route = single_shape[
                ["calitp_itp_id", "calitp_url_number", "shape_id"]
            ].iloc[
                [0]
            ]  # preserve info cols
            single_route["geometry"] = route_line
            single_route = gpd.GeoDataFrame(single_route, crs=WGS84)

            # https://stackoverflow.com/questions/15819050/pandas-dataframe-concat-vs-append/48168086
            all_routes = pd.concat(
                [all_routes, single_route], ignore_index=True, axis=0
            )

    all_routes = (
        all_routes.to_crs(CRS)
        .sort_values(["calitp_itp_id", "shape_id"])
        .drop_duplicates()
        .reset_index(drop=True)
    )

    return all_routes


In [59]:
#bay_area3 = make_routes_shapefile(ITP_ID_LIST = [4,279,282,290,294], CRS="EPSG:4326")

In [60]:
#bay_area3.head()

In [61]:
#bay_area3.to_file('bay_area_line.geojson', driver='GeoJSON')  

## San Francisco Muni Only
* Get dim stops & dim times into one df.
* [Resource 1](https://github.com/cal-itp/data-analyses/blob/main/bus_service_increase/warehouse_queries.py#L71-L76)
* [Resource 2](https://github.com/cal-itp/data-analyses/blob/main/bus_service_increase/setup_parallel_trips_with_stops.py#L136-L153)
* [Resource 3](https://docs.calitp.org/data-infra/analytics_examples/new_tutorial.html)
* Getting lots of duplicate stop ids with similar looking lat & lon again.

In [62]:
#just getting stops table first.
SF = (
    tbl.views.gtfs_schedule_dim_stops()
    >> select(_.itp_id == _.calitp_itp_id, _.stop_id, 
             _.stop_lat, _.stop_lon, _.stop_name)
    >> arrange(_.itp_id, _.stop_id, 
               _.stop_lat, _.stop_lon)
    >> collect()
    >> filter(_.itp_id == 282)
    >> distinct()
)

In [63]:
SF.shape

(9066, 5)

In [64]:
#SF = SF.drop_duplicates('stop_id')

In [65]:
SF.stop_id.nunique()

6328

### Stop Sequence
* Why do I need to get calitp deleted at > max date
* Filter out for only one trip ID, but this goes down from like 2477 unique stop IDS down to 92 unique stop ids. Seems wrong?
* GTFS_schedule_dim_stops has a lot more unique stop ids than GTFS_Schedule_stop_times?
* What does trip ID stand for?


In [66]:
#Tiffany's Code.
#    tbl_stop_times = (tbl.views.gtfs_schedule_dim_stop_times()
               #       >> filter(_.calitp_extracted_at <= min_date, 
                               # _.calitp_deleted_at > max_date)
               #       >> select(_.calitp_itp_id, _.trip_id, _.departure_time,
                           #     _.stop_sequence, _.stop_id)
   # )

In [67]:
SELECTED_DATE = "2022-02-28"
ITP_ID = 282

In [68]:
#getting times so I can grab the sequence of stops.
SF_times = (
    tbl.views.gtfs_schedule_stop_times()
    >> filter(_.calitp_itp_id == 282)
    >> filter(_.calitp_extracted_at <= SELECTED_DATE)
    >> select(_.calitp_itp_id, _.trip_id, _.departure_time,
                                _.stop_sequence, _.stop_id)
    >> collect()
    >> distinct()
)

In [69]:
SF_times.dtypes

calitp_itp_id      int64
trip_id           object
departure_time    object
stop_sequence      int64
stop_id           object
dtype: object

In [70]:
len(SF_times)

2585941

In [71]:
SF_times.stop_id.nunique()

2477

In [77]:
SF_times.sort_values(['trip_id','stop_sequence']).head(50)

,calitp_itp_id,trip_id,departure_time,stop_sequence,stop_id
2201145,282,10001444,15:07:00,1,4006
32679,282,10001444,15:10:09,2,3984
1984235,282,10001444,15:12:12,3,3987
369107,282,10001444,15:07:00,4,4006
1754216,282,10001444,15:14:19,4,6214
2211103,282,10001444,15:15:08,5,6221
2419337,282,10001444,15:15:39,6,6216
65764,282,10001444,15:17:07,7,6218
389242,282,10001444,15:10:09,7,3984
76574,282,10001444,15:12:12,8,3987


In [72]:
#filter out for only one trip id, 9987569
SF_times2 = SF_times.loc[SF_times['trip_id'] == '9987569'] 

In [73]:
SF_times2.stop_id.nunique()

92

### Merging

In [26]:
SF_times = SF_times.drop(columns=['calitp_itp_id'])

In [27]:
SF_joined = pd.merge(SF_times, SF,  how='inner', on=['stop_id'], indicator=True)

In [28]:
SF_joined._merge.value_counts()

both          307
left_only       0
right_only      0
Name: _merge, dtype: int64

In [29]:
#drop duplicates
#SF_joined2 = SF_joined.drop_duplicates(subset=['stop_id'])

In [30]:
SF_joined3 = shared_utils.geography_utils.create_point_geometry(SF_joined, 
                                                               'stop_lon', 'stop_lat')

In [31]:
SF_joined3.sort_values('stop_sequence').head(50)

,trip_id,departure_time,stop_sequence,stop_id,itp_id,stop_lat,stop_lon,stop_name,_merge,geometry
8,9987569,13:36:00,1,4648,282,37.722895,-122.394842,Fitzgerald Ave & Keith St,both,POINT (-122.39484 37.72290)
9,9987569,13:36:00,1,4648,282,37.722895,-122.394842,Fitzgerald Ave & Keith St,both,POINT (-122.39484 37.72290)
227,9987569,13:36:36,2,4647,282,37.722028,-122.393320,Fitzgerald Ave & Jennings St,both,POINT (-122.39332 37.72203)
228,9987569,13:36:36,2,4647,282,37.722028,-122.393320,Fitzgerald Ave & Jennings St,both,POINT (-122.39332 37.72203)
115,9987569,13:37:21,3,4646,282,37.720966,-122.391453,Fitzgerald Ave & Ingalls St,both,POINT (-122.39145 37.72097)
116,9987569,13:37:21,3,4646,282,37.720966,-122.391453,Fitzgerald Ave & Ingalls St,both,POINT (-122.39145 37.72097)
10,9987569,13:38:09,4,4645,282,37.719921,-122.389587,Fitzgerald Ave & Hawes St,both,POINT (-122.38959 37.71992)
11,9987569,13:38:09,4,4645,282,37.719921,-122.389587,Fitzgerald Ave & Hawes St,both,POINT (-122.38959 37.71992)
70,9987569,13:39:10,5,4784,282,37.718234,-122.388272,Gilman Ave & Griffith St,both,POINT (-122.38827 37.71823)
69,9987569,13:39:10,5,4784,282,37.718234,-122.388272,Gilman Ave & Griffith St,both,POINT (-122.38827 37.71823)


In [39]:
SF_joined4 = SF_joined3.drop_duplicates(subset=['stop_id'])

In [42]:
SF_joined4.sort_values('stop_sequence').head()

,trip_id,departure_time,stop_sequence,stop_id,itp_id,stop_lat,stop_lon,stop_name,_merge,geometry
8,9987569,13:36:00,1,4648,282,37.722895,-122.394842,Fitzgerald Ave & Keith St,both,POINT (-122.39484 37.72290)
227,9987569,13:36:36,2,4647,282,37.722028,-122.393320,Fitzgerald Ave & Jennings St,both,POINT (-122.39332 37.72203)
115,9987569,13:37:21,3,4646,282,37.720966,-122.391453,Fitzgerald Ave & Ingalls St,both,POINT (-122.39145 37.72097)
10,9987569,13:38:09,4,4645,282,37.719921,-122.389587,Fitzgerald Ave & Hawes St,both,POINT (-122.38959 37.71992)
69,9987569,13:39:10,5,4784,282,37.718234,-122.388272,Gilman Ave & Griffith St,both,POINT (-122.38827 37.71823)


In [45]:
#SF_joined4.to_file('SF_joined4.geojson', driver='GeoJSON')  

/opt/conda/lib/python3.9/site-packages/geopandas/io/file.py:362: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
